# Das Merit-Order-Modell – ein interaktives Notebook
**Kursmaterial zum Modul 32941 – Kurseinheit *Nachhaltige Energiewirtschaft***

Dieses Notebook ergänzt Kapitel 2 des Lehrtexts. Es baut das Merit-Order-Modell schrittweise auf und reproduziert die dort verwendeten Tabellen und Abbildungen. Über Regler können Sie Nachfrage, erneuerbare Einspeisung und CO₂-Preis verändern und unmittelbar beobachten, wie sich Kraftwerkseinsatz, Marktpreis und Renten anpassen.

**Überblick**

| Abschnitt | Thema | Bezug zum Lehrtext |
|---|---|---|
| 2 | Kraftwerkspark und Nachfrage | Tabelle 7 |
| 3 | Grenzkosten und CO₂-Preis | Abschnitt 2.2.1 |
| 4 | Merit Order und Marktpreis | Abschnitt 2.2.3 |
| 5 | Angebotskurve und Markträumung | Abbildung 11 |
| 6 | Einsatz, Kosten und inframarginale Renten | Tabelle 8 |
| 7 | Merit-Order-Effekt | Abbildung 12 |
| 8 | Preise ohne und mit erneuerbarer Einspeisung | Tabelle 9 |
| 9 | CO₂-Preis und Brennstoffwechsel | Abbildung 13 |
| 10 | Marktwert und Selbstkannibalisierung | Abbildung 14 |

Führen Sie die Zellen in der vorgegebenen Reihenfolge aus. Vorkenntnisse in Python sind nicht erforderlich. Hinweise zur Installation und Ausführung enthält die README-Datei im [GitHub-Kursraum](https://github.com/fuh-energy/Nachhaltige-Energiewirtschaft).

**Hinweis zum Verfahren.** Zur Lösung des hier verwendeten Merit-Order-Modells ist kein Optimierungslöser (sogenannter Solver, wie bspw. HiGHS, CPLEX oder Gurobi) erforderlich. Die Kraftwerke werden nach aufsteigenden Grenzkosten sortiert und ihre verfügbaren Leistungen kumuliert. Preissetzend ist die günstigste Anlage, die noch über freie Kapazität zur Bereitstellung einer zusätzlichen Megawattstunde Strom verfügt.


## 1  Vorbereitung

Führen Sie zunächst die folgende Zelle aus. Sie lädt die benötigten Funktionen und bereitet die interaktiven Regler vor. Falls eine Fehlermeldung auf ein fehlendes Programmpaket hinweist, folgen Sie bitte der Installationsanleitung in der README-Datei des GitHub-Kursraums. Ohne die Erweiterung für interaktive Regler bleibt das Notebook lauffähig und zeigt stattdessen jeweils das im Lehrtext verwendete Ausgangsbeispiel.


In [ ]:
# Falls bei der Ausführung ein Paket fehlt, folgen Sie bitte der README-Datei.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from decimal import Decimal, ROUND_HALF_EVEN

try:
    from ipywidgets import interact, IntSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("Die interaktiven Regler sind nicht verfügbar. Stattdessen wird jeweils "
          "das Ausgangsbeispiel des Lehrtexts berechnet.")


def de(v, dec=2):
    """Formatiert eine Zahl mit deutschem Dezimal- und Tausendertrennzeichen."""
    if pd.isna(v):
        return "–"
    quant = Decimal("1").scaleb(-dec)
    wert = Decimal(f"{float(v):.12g}").quantize(quant, rounding=ROUND_HALF_EVEN)
    return f"{wert:,.{dec}f}".replace(",", "#").replace(".", ",").replace("#", ".")


pd.set_option("display.float_format", de)


def interaktiv(fn, festes_beispiel=None, **regler):
    """Zeigt Regler oder berechnet ersatzweise das festgelegte Ausgangsbeispiel."""
    if HAS_WIDGETS:
        widgets = {
            name: IntSlider(min=lo, max=hi, step=st, value=val,
                            description=lab,
                            style={"description_width": "150px"},
                            continuous_update=False)
            for name, (lo, hi, st, val, lab) in regler.items()
        }
        interact(fn, **widgets)
    else:
        print("(Statische Ausgabe des Ausgangsbeispiels)\n")
        fn(**(festes_beispiel or {}))


print("Vorbereitung abgeschlossen.")


## 2  Kraftwerkspark und Nachfrage (Tabelle 7)

Das durchgängige Zahlenbeispiel umfasst neun Kraftwerke. Zu jeder Anlage werden vier Größen hinterlegt:

- `capacity` bezeichnet die verfügbare Leistung in MW.
- `c_var0` fasst den CO₂-unabhängigen Teil der Grenzkosten zusammen. Er enthält die Brennstoffkosten und die sonstigen variablen Betriebskosten.
- `ef` bezeichnet den CO₂-Emissionsfaktor in Tonnen CO₂ je MWh Strom.
- `colour` legt ausschließlich die Farbe der Anlage in den Abbildungen fest.

Die Aufteilung in `c_var0` und `ef` ermöglicht es, den CO₂-Preis in den folgenden Abschnitten zu verändern. Die Werte entsprechen dem Kraftwerkspark aus Tabelle 7. Die Emissionsfaktoren sind didaktisch gewählt. Für Biomasse wird entsprechend der vereinfachten bilanziellen Behandlung im Lehrtext ein Emissionsfaktor von null angesetzt.

Zusätzlich werden die Nachfrage in sechs vierstündigen Zeitschritten und der CO₂-Preis des Ausgangsfalls von 25 €/tCO₂ eingelesen. Dieselben Zahlen und Annahmen liegen auch dem GAMS-Modell *BasisModell.gms* zugrunde.


In [ ]:
# Eine Zeile je Kraftwerk. Der Index ist der im Code verwendete Kurzname.
TECH = pd.DataFrame({
    "label":    ["Laufwasser", "Kernkraft", "Braunkohle", "Gas GuD", "Steinkohle 1",
                 "Gas 1", "Steinkohle 2", "Biomasse", "Gas 2"],
    "capacity": [50, 1000, 800, 700, 750, 200, 650, 20, 150],                    # MW
    "c_var0":   [0.00, 6.06, 12.25, 34.11, 32.15, 49.00, 42.61, 66.82, 67.50],  # €/MWh, ohne CO₂
    "ef":       [0.00, 0.00, 1.05, 0.35, 0.80, 0.50, 0.80, 0.00, 0.50],          # tCO₂/MWh_el
    "colour":   ["#3182bd", "#c724b1", "#7b5141", "#fdae6b", "#333333",
                 "#e6550d", "#7a7a7a", "#31a354", "#fb9a29"],
}, index=["laufwasser", "kernkraft", "braunkohle", "gud", "steinkohle_1",
          "gas_1", "steinkohle_2", "biomasse", "gas_2"])

# Nachfrage und Dauer der sechs Zeitschritte des durchgängigen Zahlenbeispiels
DEMAND = pd.Series([3000, 3500, 4200, 3900, 4000, 3600],
                   index=[f"t{i+1}" for i in range(6)], name="Nachfrage (MW)")
DT = 4              # Stunden je Zeitschritt
CO2_BASE = 25.0     # €/tCO₂

# Farben für die erneuerbare Einspeisung ab Abschnitt 7
VRE_COLOUR = {"Wind": "#74c476", "PV": "#fdd835"}

eingabedaten = pd.DataFrame({
    "Anlage": TECH["label"],
    "Verfügbare Leistung (MW)": TECH["capacity"],
    "CO₂-unabhängige Grenzkosten (€/MWh)": TECH["c_var0"],
    "Emissionsfaktor (tCO₂/MWh_el)": TECH["ef"],
}).reset_index(drop=True)
display(eingabedaten)


## 3  Grenzkosten und CO₂-Preis (Abschnitt 2.2.1)

Die kurzfristigen Grenzkosten einer thermischen Anlage $i$ setzen sich aus Brennstoffkosten, sonstigen variablen Betriebskosten und CO₂-Kosten zusammen:

$$
c_i = \frac{p_i^{\mathrm{Brennstoff}}}{\eta_i} + voc_i + ef_i \cdot p^{\mathrm{CO_2}}
$$

Die Brennstoffkosten und die sonstigen variablen Betriebskosten werden in den Eingabedaten als $c_i^{\mathrm{var},0}$ zusammengefasst. Damit gilt:

$$
c_i = c_i^{\mathrm{var},0} + ef_i \cdot p^{\mathrm{CO_2}}
$$

Ein höherer CO₂-Preis verteuert die Kraftwerke daher nicht gleichmäßig. Anlagen mit einem höheren Emissionsfaktor werden stärker belastet. Dieser Zusammenhang bildet die Grundlage des in Abschnitt 9 untersuchten Brennstoffwechsels.


In [ ]:
def grenzkosten(co2_preis=CO2_BASE):
    """Grenzkosten aller Kraftwerke in €/MWh bei einem gegebenen CO₂-Preis in €/tCO₂."""
    return TECH["c_var0"] + TECH["ef"] * co2_preis


# Vergleich der Grenzkosten bei 25 und 80 €/tCO₂
kontrolle = pd.DataFrame({
    "Anlage": TECH["label"],
    "Grenzkosten bei 25 €/tCO₂ (€/MWh)": grenzkosten(CO2_BASE),
    "Grenzkosten bei 80 €/tCO₂ (€/MWh)": grenzkosten(80),
}).reset_index(drop=True)
display(kontrolle.sort_values("Grenzkosten bei 25 €/tCO₂ (€/MWh)"))

# Zerlegung der Grenzkosten des Braunkohle- und des Gas-und-Dampf-Kraftwerks
print("Zerlegung der Grenzkosten bei 25 €/tCO₂:\n")
for name in ["braunkohle", "gud"]:
    r = TECH.loc[name]
    co2_anteil = r["ef"] * CO2_BASE
    print(f"{r['label']:12s}: {de(r['c_var0'] + co2_anteil)} €/MWh = "
          f"{de(r['c_var0'])} €/MWh (Brennstoff und sonstige variable Kosten) + "
          f"{de(co2_anteil)} €/MWh (CO₂-Zertifikate)")


## 4  Merit Order und Marktpreis (Abschnitt 2.2.3)

Sind die Anlagen aufsteigend nach Grenzkosten geordnet, ist das preissetzende Kraftwerk $m(t)$ die erste Anlage, deren Einbeziehung dazu führt, dass die kumulierte verfügbare Leistung die Nachfrage übersteigt. Der Marktpreis entspricht den Grenzkosten dieser Anlage:

$$
m(t)=\min\left\{k \in \{1,\ldots,n\}\;\middle|\;\sum_{i=1}^{k} y_i^{\max}>D_t\right\},
\qquad
p_t=c_{m(t)}
$$

Die Berechnung erfolgt in vier Schritten:

1. Die Anlagen werden nach aufsteigenden Grenzkosten sortiert.
2. Ihre verfügbaren Leistungen werden entlang dieser Reihenfolge kumuliert.
3. Bestimmt wird die erste Anlage, mit der die kumulierte Leistung die Nachfrage übersteigt. Sie ist die günstigste Anlage mit freier Kapazität für eine zusätzliche Megawattstunde.
4. Ihre Grenzkosten bestimmen den Marktpreis.

Im Code übernehmen `sort_values()` und `cumsum()` die ersten beiden Schritte.

Die strikte Ungleichung $>$ ist relevant, wenn die Nachfrage genau einer Kapazitätsgrenze entspricht. Das Notebook verwendet dann entsprechend der im Lehrtext gewählten Konvention die Grenzkosten der nächsten Anlage mit freier Kapazität. Der Zeitschritt t2 bildet einen solchen Grenzfall. Sind sämtliche verfügbaren Kapazitäten vollständig ausgelastet, kann diese Regel dagegen keinen Marktpreis mehr bestimmen. Dazu wäre eine zusätzliche Knappheits- oder Lastabwurfoption erforderlich.


In [ ]:
def merit_order(co2_preis=CO2_BASE, wind=0.0, pv=0.0):
    """Ordnet Anlagen nach Grenzkosten und kumuliert ihre verfügbaren Leistungen."""
    rows = pd.DataFrame({
        "label":    TECH["label"],
        "capacity": TECH["capacity"].astype(float),
        "mc":       grenzkosten(co2_preis),
        "ef":       TECH["ef"],
        "colour":   TECH["colour"],
    })

    # Wind und PV werden nur ergänzt, wenn eine positive Einspeisung vorgegeben ist.
    extra = []
    if wind > 0:
        extra.append(pd.DataFrame({"label": ["Wind"], "capacity": [float(wind)],
                                   "mc": [0.0], "ef": [0.0],
                                   "colour": [VRE_COLOUR["Wind"]]}, index=["wind"]))
    if pv > 0:
        extra.append(pd.DataFrame({"label": ["PV"], "capacity": [float(pv)],
                                   "mc": [0.0], "ef": [0.0],
                                   "colour": [VRE_COLOUR["PV"]]}, index=["pv"]))
    if extra:
        rows = pd.concat([rows] + extra)

    rows = rows.sort_values("mc", kind="stable")
    rows["cum_before"] = rows["capacity"].cumsum() - rows["capacity"]
    rows["cum_after"] = rows["capacity"].cumsum()
    return rows


def marktpreis(nachfrage, co2_preis=CO2_BASE, wind=0.0, pv=0.0):
    """Bestimmt Marktpreis und preissetzende Anlage für eine gegebene Nachfrage.

    Bei einer Blockgrenze gilt die Konvention des Lehrtexts: Die nächste Anlage
    mit freier Kapazität setzt den Preis. Bei vollständiger Auslastung des gesamten
    Kraftwerksparks ist der Knappheitspreis im vereinfachten Modell nicht bestimmt.
    """
    mo = merit_order(co2_preis, wind, pv)
    gesamt = float(mo["cum_after"].max())

    if nachfrage > gesamt + 1e-9:
        return np.nan, "Nachfrage übersteigt die verfügbare Gesamtleistung"
    if abs(nachfrage - gesamt) <= 1e-9:
        return np.nan, "Gesamte verfügbare Leistung ist ausgelastet"

    frei = mo[mo["cum_after"] > nachfrage + 1e-9]
    row = frei.iloc[0]
    return float(row["mc"]), row["label"]


# Merit Order des Ausgangsfalls – vergleichen Sie die Ausgabe mit Tabelle 7.
mo = merit_order()
merit_order_ausgabe = pd.DataFrame({
    "Anlage": mo["label"],
    "Verfügbare Leistung (MW)": mo["capacity"],
    "Grenzkosten (€/MWh)": mo["mc"],
    "Kumulierte Leistung (MW)": mo["cum_after"],
    "Emissionsfaktor (tCO₂/MWh_el)": mo["ef"],
}).reset_index(drop=True)
display(merit_order_ausgabe)

# Marktpreis und preissetzende Anlage in den sechs Zeitschritten
print("Markträumung in den sechs Zeitschritten:\n")
for t, d in DEMAND.items():
    p, kraftwerk = marktpreis(d)
    print(f"{t}: Nachfrage {d:5.0f} MW  →  Preis {de(p):>6s} €/MWh   "
          f"(preissetzend: {kraftwerk})")


## 5  Angebotskurve und Markträumung (Abbildung 11)

Jeder Kraftwerksblock ist so breit wie die verfügbare Leistung der Anlage und so hoch wie ihre Grenzkosten. Werden die Blöcke in der Reihenfolge der Merit Order aneinandergesetzt, entsteht die treppenförmige Angebotskurve. Die senkrechte gestrichelte Linie zeigt die als preisunelastisch angenommene Marktnachfrage. Die waagerechte gepunktete Linie kennzeichnet den Marktpreis.

Die folgende Funktion erzeugt Abbildung 11 des Lehrtexts.


In [ ]:
def abbildung_11(nachfrage=4200, co2_preis=CO2_BASE, wind=0.0, pv=0.0,
                 ax=None, ymax=None, save_as=None):
    """Zeichnet Angebotskurve, Marktnachfrage und Marktpreis wie in Abbildung 11."""
    mo = merit_order(co2_preis, wind, pv)
    if ymax is None:
        ymax = max(105, mo["mc"].max() * 1.30)
    preis, kraftwerk = marktpreis(nachfrage, co2_preis, wind, pv)
    grau, farbe_preis = "#666666", "#e6550d"

    if ax is None:
        fig, ax = plt.subplots(figsize=(9.5, 5.2))
    else:
        fig = ax.figure

    # Breite eines Blocks = verfügbare Leistung, Höhe = Grenzkosten
    for _, r in mo.iterrows():
        ax.bar(r["cum_before"], r["mc"], width=r["capacity"], align="edge",
               color=r["colour"], edgecolor="white", linewidth=0.7,
               label=r["label"], zorder=3)

    ax.axvline(nachfrage, color=grau, linestyle="--", linewidth=1.4, zorder=4)
    ax.text(nachfrage - 70, ymax * 0.962, "Marktnachfrage D", color=grau,
            fontsize=9.5, ha="right", va="top")

    if np.isfinite(preis):
        ax.plot([0, nachfrage], [preis, preis], color=farbe_preis,
                linestyle=":", linewidth=1.9, zorder=4)
        ax.text(nachfrage * 0.63, preis - 3,
                f"Marktpreis p = {de(preis)} €/MWh\n({kraftwerk} ist preissetzend)",
                color=farbe_preis, fontsize=9.5, va="top", ha="left")

    gesamt = mo["cum_after"].max()
    ax.set_xlim(0, max(gesamt, nachfrage) * 1.01)
    ax.set_ylim(0, ymax)
    ax.set_xlabel("Kumulierte verfügbare Leistung (MW)", fontsize=10)
    ax.set_ylabel("Grenzkosten, Preis (€/MWh)", fontsize=10)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.set_facecolor("white")

    legend = ax.legend(loc="upper left", bbox_to_anchor=(0.015, 0.975), frameon=True,
                       fontsize=8.5, borderaxespad=0.0, fancybox=True,
                       markerfirst=False, handletextpad=0.7, ncol=3, columnspacing=1.2)
    frame = legend.get_frame()
    frame.set_facecolor("white"); frame.set_edgecolor("#bdbdbd")
    frame.set_linewidth(0.7); frame.set_alpha(0.96)
    set_boxstyle = getattr(frame, "set_boxstyle", None)
    if callable(set_boxstyle):
        set_boxstyle("round,pad=0.35,rounding_size=0.18")
    frame.set_path_effects([pe.SimplePatchShadow(offset=(1.5, -1.5), alpha=0.18,
                                                 shadow_rgbFace="#999999"), pe.Normal()])
    for txt in legend.get_texts():
        txt.set_horizontalalignment("right")
    if hasattr(legend, "set_alignment"):
        legend.set_alignment("right")

    fig.tight_layout()
    if save_as:
        fig.savefig(save_as, bbox_inches="tight")
        print("Gespeichert unter", save_as)
    return ax


abbildung_11(nachfrage=4200)
plt.show()


### Verändern Sie Nachfrage und CO₂-Preis

Verschieben Sie die Nachfragelinie mit dem Regler **Nachfrage**. Beobachten Sie, welche Anlage noch über freie Kapazität verfügt und damit den Marktpreis bestimmt. Verändern Sie anschließend den **CO₂-Preis**. Dadurch ändern sich die Höhe und gegebenenfalls die Reihenfolge der Kraftwerksblöcke.

Vergleichen Sie außerdem Nachfragewerte von 3.299 und 3.301 MW. Sie können den Wert direkt in das Eingabefeld des Reglers schreiben. Die Erhöhung um lediglich 2 MW lässt den Marktpreis von 52,15 auf 61,50 €/MWh springen, weil die verfügbare Leistung von Steinkohle 1 überschritten wird und Gas 1 preissetzend wird. Der Preissprung ist eine Folge der stufenförmigen Angebotskurve.


In [ ]:
def zeige_angebotskurve(nachfrage=4200, co2_preis=25):
    abbildung_11(nachfrage=nachfrage, co2_preis=co2_preis)
    plt.show()

    p, kraftwerk = marktpreis(nachfrage, co2_preis)
    gesamt = merit_order(co2_preis)["cum_after"].max()
    if np.isfinite(p):
        print(f"Preis {de(p)} €/MWh, preissetzend: {kraftwerk}.")
    elif nachfrage > gesamt:
        print(f"Die Nachfrage übersteigt die verfügbare Gesamtleistung von {de(gesamt, 0)} MW. "
              "Ohne Lastabwurf kann der Markt nicht geräumt werden.")
    else:
        print(f"Die Nachfrage entspricht der verfügbaren Gesamtleistung von {de(gesamt, 0)} MW. "
              "Sie kann genau gedeckt werden, der Knappheitspreis ist im vereinfachten Modell "
              "jedoch nicht bestimmt.")

interaktiv(zeige_angebotskurve,
           festes_beispiel=dict(nachfrage=4200, co2_preis=25),
           nachfrage=(500, 5000, 100, 4200, "Nachfrage (MW)"),
           co2_preis=(0, 150, 5, 25, "CO₂-Preis (€/t)"))


**Ausgangsfall.** Bei einer Nachfrage von 4.200 MW werden die Anlagen bis einschließlich Biomasse vollständig ausgelastet. Gas 2 liefert die noch fehlenden 30 MW und verfügt danach über 120 MW freie Kapazität. Gas 2 ist somit preissetzend und der Marktpreis beträgt 80,00 €/MWh.

Setzen Sie die Nachfrage anschließend auf 4.000, 3.000, 1.500 und 1.000 MW. Vergleichen Sie die resultierenden Preise und preissetzenden Anlagen mit der Sensitivitätsanalyse in Abschnitt 2.2.4 des Lehrtexts.


## 6  Einsatz, Kosten und inframarginale Renten (Tabelle 8)

Bei der Einheitspreisbildung erhält jede eingesetzte Anlage denselben Marktpreis. Liegen ihre Grenzkosten unter diesem Preis, erzielt sie eine inframarginale Rente. Dabei bezeichnet $p_t$ den Marktpreis, $mc_i$ die Grenzkosten, $g_{i,t}$ die Erzeugungsleistung und $\Delta t$ die Dauer des Zeitschritts:

$$
R_{i,t}=(p_t-mc_i)\cdot g_{i,t}\cdot\Delta t
$$

Die inframarginale Rente ist kein Gewinn. Sie bildet zunächst einen Deckungsbeitrag zur Finanzierung der in den Grenzkosten nicht enthaltenen Investitions- und sonstigen Fixkosten. Im betrachteten Ausgangsfall erzielt Gas 2 als preissetzende Anlage keinen Deckungsbeitrag, weil der Marktpreis genau seinen Grenzkosten entspricht.

Die folgende Zelle reproduziert Tabelle 8 für den dritten Zeitschritt mit einer Nachfrage von 4.200 MW und einer Dauer von vier Stunden.


In [ ]:
def einsatz_und_renten(nachfrage, co2_preis=CO2_BASE, wind=0.0, pv=0.0):
    """Berechnet Erzeugung, variable Kosten und inframarginale Rente je Anlage."""
    mo = merit_order(co2_preis, wind, pv)
    preis, preissetzend = marktpreis(nachfrage, co2_preis, wind, pv)
    if not np.isfinite(preis):
        raise ValueError("Der Marktpreis ist bei dieser Nachfrage im vereinfachten Modell nicht bestimmt.")

    erzeugung = (nachfrage - mo["cum_before"]).clip(lower=0)
    erzeugung = pd.concat([erzeugung, mo["capacity"]], axis=1).min(axis=1)

    out = pd.DataFrame({
        "Anlage": mo["label"],
        "Grenzkosten": mo["mc"],
        "Erzeugung": erzeugung,
        "Variable Kosten": mo["mc"] * erzeugung * DT,
        "Inframarginale Rente": (preis - mo["mc"]) * erzeugung * DT,
    })
    out = out[out["Erzeugung"] > 0].copy()
    return out, preis, preissetzend


tabelle, preis, preissetzend = einsatz_und_renten(4200)
var_kosten = tabelle["Variable Kosten"].sum()
zahlung = preis * 4200 * DT
produzentenrente = tabelle["Inframarginale Rente"].sum()

# Ausgabe in derselben Form wie Tabelle 8 des Lehrtexts
ausgabe = pd.DataFrame({
    "Anlage": [f"{a} (preissetzend)" if a == preissetzend else a for a in tabelle["Anlage"]],
    "Grenzkosten (€/MWh)": [de(v) for v in tabelle["Grenzkosten"]],
    "Erzeugung (MW)": [de(v, 0) for v in tabelle["Erzeugung"]],
    "Variable Kosten (€)": [de(v, 0) for v in tabelle["Variable Kosten"]],
    "Inframarginale Rente (€)": [de(v, 0) for v in tabelle["Inframarginale Rente"]],
})
ausgabe.loc[len(ausgabe)] = ["Summe", "–", de(tabelle["Erzeugung"].sum(), 0),
                             de(var_kosten, 0), de(produzentenrente, 0)]
display(ausgabe)

print(f"Marktpreis                                      : {de(preis)} €/MWh")
print(f"Erzeugung insgesamt                             : {de(tabelle['Erzeugung'].sum(), 0)} MW")
print(f"Variable Erzeugungskosten                       : {de(var_kosten, 0)} €")
print(f"Verbraucherzahlung                              : {de(zahlung, 0)} €")
print(f"Produzentenrente                                : {de(produzentenrente, 0)} €")
print(f"Summe der inframarginalen Renten                : {de(tabelle['Inframarginale Rente'].sum(), 0)} €")

gas2 = tabelle.loc[tabelle["Anlage"] == "Gas 2"].iloc[0]
kernkraft = tabelle.loc[tabelle["Anlage"] == "Kernkraft"].iloc[0]
anteil_kernkraft = kernkraft["Inframarginale Rente"] / produzentenrente * 100
print(f"\nGas 2 erzeugt {de(gas2['Erzeugung'], 0)} MW und erzielt eine inframarginale Rente von "
      f"{de(gas2['Inframarginale Rente'], 0)} €.")
print(f"Das Kernkraftwerk erzielt {de(kernkraft['Inframarginale Rente'], 0)} €. "
      f"Das entspricht {de(anteil_kernkraft, 1)} % der Produzentenrente.")

assert np.isclose(tabelle["Erzeugung"].sum(), 4200)
assert np.isclose(produzentenrente, zahlung - var_kosten)


## 7  Merit-Order-Effekt erneuerbarer Einspeisung (Abbildung 12)

Windenergie und Photovoltaik weisen im vereinfachten Modell Grenzkosten nahe null auf. Ihre verfügbare Einspeisung wird deshalb am linken Rand der Merit Order eingeordnet. Bei unveränderten Brennstoff- und CO₂-Preisen bleibt die Reihenfolge der konventionellen Anlagen bestehen. Der konventionelle Teil der Angebotskurve verschiebt sich lediglich um die erneuerbare Einspeisung nach rechts.

Bei unveränderter Nachfrage kann dadurch eine Anlage mit niedrigeren Grenzkosten preissetzend werden. Die resultierende Preissenkung wird als **Merit-Order-Effekt** bezeichnet.

Abbildung 12 zeigt die Angebotskurven als Treppenfunktionen. Da die Blöcke von Wind und Photovoltaik unmittelbar auf der Abszisse kaum zu erkennen wären, kennzeichnet ein grüner Doppelpfeil die erneuerbare Einspeisung und damit die horizontale Verschiebung.


In [ ]:
def _treppenkurve(mo, offset=0.0):
    """Wandelt eine Merit-Order-Tabelle in die Punkte einer Treppenkurve um."""
    xs, ys = [], []
    x = offset
    for _, r in mo.iterrows():
        xs += [x, x + r["capacity"]]
        ys += [r["mc"], r["mc"]]
        x += r["capacity"]
    return xs, ys


def abbildung_12(nachfrage=4200, wind=1200, pv=800, co2_preis=CO2_BASE,
                 ymax=None, save_as=None):
    """Zeichnet den Merit-Order-Effekt wie in Abbildung 12."""
    ee = wind + pv
    konv = merit_order(co2_preis)
    if ymax is None:
        ymax = max(105, konv["mc"].max() * 1.30)
    p_ohne, kw_ohne = marktpreis(nachfrage, co2_preis)
    p_mit, kw_mit = marktpreis(nachfrage, co2_preis, wind, pv)

    x0, y0 = _treppenkurve(konv, 0.0)
    x1, y1 = _treppenkurve(konv, ee)

    grau, c_kurve, c_effekt, c_ee = "#666666", "#08519c", "#e6550d", "#74c476"
    fig, ax = plt.subplots(figsize=(9.5, 5.2))

    ax.plot(x0, y0, color=grau, linestyle="--", linewidth=1.8, zorder=3)
    ax.plot(x1, y1, color=c_kurve, linestyle="-", linewidth=2.4, zorder=4)

    if ee > 0:
        ax.annotate("", xy=(0, 6.5), xytext=(ee, 6.5),
                    arrowprops=dict(arrowstyle="<->", color=c_ee, linewidth=2.0))
        ax.text(ee / 2, 8.0, f"Wind + PV: {de(ee, 0)} MW zu Grenzkosten nahe null",
                color="#2f7d43", fontsize=9.5, ha="center", va="bottom")
        ax.axvline(ee, color=c_ee, linestyle=":", linewidth=1.4, zorder=2)

    ax.axvline(nachfrage, color=grau, linestyle="--", linewidth=1.4, zorder=3)
    ax.text(nachfrage - 90, ymax * 0.962, "Marktnachfrage D",
            color=grau, fontsize=9.5, ha="right", va="top")

    ax.plot([0, nachfrage], [p_ohne, p_ohne], color=grau, linestyle=":", linewidth=1.6, zorder=3)
    ax.plot([0, nachfrage], [p_mit, p_mit], color=c_effekt, linestyle=":", linewidth=1.9, zorder=4)
    ax.plot([nachfrage], [p_ohne], marker="o", color=grau, markersize=7, zorder=6)
    ax.plot([nachfrage], [p_mit], marker="o", color=c_effekt, markersize=7, zorder=6)

    xa = nachfrage + 350
    if abs(p_mit - p_ohne) > 1e-9:
        ann = ax.annotate("", xy=(xa, p_mit), xytext=(xa, p_ohne),
                          arrowprops=dict(arrowstyle="->", color=c_effekt, linewidth=3.0),
                          zorder=11)
        if ann.arrow_patch is not None:
            ann.arrow_patch.set_zorder(10)
        ax.text(xa + 30, (p_mit + p_ohne) / 2 + 5,
                f"Merit-Order-Effekt\nΔp = {de(p_mit - p_ohne)} €/MWh",
                color=c_effekt, fontsize=9.5, ha="left", va="center")
    else:
        ax.text(xa + 60, p_ohne, "Kein Preiseffekt\nbei dieser Einspeisung",
                color=grau, fontsize=9.5, ha="left", va="center")

    ax.text(120, p_ohne + 1.5, f"p = {de(p_ohne)} €/MWh ({kw_ohne} preissetzend)",
            color=grau, fontsize=9.5, ha="left", va="bottom")
    if abs(p_mit - p_ohne) > 1e-9:
        ax.text(120, p_mit + 1.5, f"p = {de(p_mit)} €/MWh ({kw_mit} preissetzend)",
                color=c_effekt, fontsize=9.5, ha="left", va="bottom")

    ax.set_xlim(0, konv["cum_after"].max() + ee + 1130)
    ax.set_ylim(0, ymax)
    ax.set_xlabel("Kumulierte verfügbare Leistung (MW)", fontsize=10)
    ax.set_ylabel("Grenzkosten, Preis (€/MWh)", fontsize=10)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.set_facecolor("white")

    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], color=grau, linestyle="--", linewidth=1.8,
                      label="Angebotskurve ohne erneuerbare Einspeisung"),
               Line2D([0], [0], color=c_kurve, linestyle="-", linewidth=2.4,
                      label="Angebotskurve mit Wind und PV")]
    legend = ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.985, 0.975),
                       frameon=True, fontsize=9, borderaxespad=0.0, fancybox=True,
                       markerfirst=False, handletextpad=0.8)
    frame = legend.get_frame()
    frame.set_facecolor("white"); frame.set_edgecolor("#bdbdbd")
    frame.set_linewidth(0.7); frame.set_alpha(0.96)
    set_boxstyle = getattr(frame, "set_boxstyle", None)
    if callable(set_boxstyle):
        set_boxstyle("round,pad=0.35,rounding_size=0.18")
    frame.set_path_effects([pe.SimplePatchShadow(offset=(1.5, -1.5), alpha=0.18,
                                                 shadow_rgbFace="#999999"), pe.Normal()])
    for txt in legend.get_texts():
        txt.set_horizontalalignment("right")
    if hasattr(legend, "set_alignment"):
        legend.set_alignment("right")

    fig.tight_layout()
    if save_as:
        fig.savefig(save_as, bbox_inches="tight")
        print("Gespeichert unter", save_as)
    return ax


abbildung_12(nachfrage=4200, wind=1200, pv=800)
plt.show()


### Verändern Sie die erneuerbare Einspeisung

Erhöhen Sie **Wind** und **PV** schrittweise. Beobachten Sie, wie sich die blaue Angebotskurve nach rechts verschiebt, während die graue Referenzkurve unverändert bleibt. Die ausgegebene Preisänderung zeigt den Merit-Order-Effekt.

Zwei Beobachtungen sind besonders wichtig:

- Der Marktpreis sinkt nicht kontinuierlich. Er verändert sich erst, wenn die erneuerbare Einspeisung eine Stufengrenze der Angebotskurve überschreitet.
- Setzen Sie die Photovoltaikeinspeisung auf 0 MW und die Nachfrage auf 3.000 MW. Vergleichen Sie anschließend das Marktergebnis bei 450 und 500 MW Windenergie. Erklären Sie den Preissprung mithilfe der kumulierten Kraftwerksleistungen aus Abschnitt 4.


In [ ]:
def zeige_mo_effekt(wind=1200, pv=800, nachfrage=4200, co2_preis=25):
    abbildung_12(nachfrage=nachfrage, wind=wind, pv=pv, co2_preis=co2_preis)
    plt.show()
    p0, kw0 = marktpreis(nachfrage, co2_preis)
    p1, kw1 = marktpreis(nachfrage, co2_preis, wind, pv)
    print(f"Ohne erneuerbare Einspeisung : {de(p0):>6s} €/MWh   (preissetzend: {kw0})")
    print(f"Mit erneuerbarer Einspeisung : {de(p1):>6s} €/MWh   (preissetzend: {kw1})")
    print(f"Preisänderung Δp             : {de(p1 - p0):>6s} €/MWh")

interaktiv(zeige_mo_effekt,
           festes_beispiel=dict(wind=1200, pv=800, nachfrage=4200, co2_preis=25),
           wind=(0, 4000, 50, 1200, "Wind (MW)"),
           pv=(0, 4000, 50, 800, "PV (MW)"),
           nachfrage=(500, 5000, 100, 4200, "Nachfrage (MW)"),
           co2_preis=(0, 150, 5, 25, "CO₂-Preis (€/t)"))


## 8  Preise ohne und mit erneuerbarer Einspeisung (Tabelle 9)

Tabelle 9 überträgt eine erneuerbare Einspeisung von 1.200 MW Windenergie und 800 MW Photovoltaik auf alle sechs Zeitschritte des durchgängigen Zahlenbeispiels. Jeder Zeitschritt dauert vier Stunden. Das arithmetische Mittel entspricht daher zugleich dem zeitgewichteten Durchschnittspreis.

Die Spalte **Differenz** gibt die Preisänderung gegenüber dem Fall ohne erneuerbare Einspeisung an. Obwohl in jedem Zeitschritt dieselbe zusätzliche Leistung eingespeist wird, liegt die Preissenkung zwischen 19,75 und 46,09 €/MWh. Der Zeitschritt t2 bildet einen Grenzfall. Entsprechend der Konvention aus Abschnitt 4 verwendet das Notebook dort die Grenzkosten der nächsten Anlage mit freier Kapazität.

### Weshalb der Effekt zwischen den Zeitschritten variiert

In t1 fällt der Marktpreis von 52,15 auf 6,06 €/MWh. Die preissetzende Anlage wechselt von Steinkohle 1 zur Kernkraft. In t4 sinkt der Preis dagegen lediglich von 62,61 auf 42,86 €/MWh und das preissetzende Kraftwerk wechselt von Steinkohle 2 zum Gas-und-Dampf-Kraftwerk. Die Höhe des Merit-Order-Effekts hängt somit von der erneuerbaren Einspeisung, der Nachfrage und den Grenzkostenabständen zwischen den betroffenen Kraftwerken ab.


In [ ]:
def preistabelle(wind=1200, pv=800, co2_preis=CO2_BASE):
    """Vergleicht die Marktpreise aller Zeitschritte ohne und mit erneuerbarer Einspeisung."""
    rows = []
    for t, d in DEMAND.items():
        p0, _ = marktpreis(d, co2_preis)
        p1, _ = marktpreis(d, co2_preis, wind, pv)
        rows.append({
            "Zeitschritt": t,
            "Nachfrage (MW)": d,
            "Preis ohne erneuerbare Einspeisung (€/MWh)": p0,
            "Preis mit Wind und PV (€/MWh)": p1,
            "Differenz (€/MWh)": p1 - p0,
        })
    tab = pd.DataFrame(rows).set_index("Zeitschritt")
    tab.loc["Mittel"] = tab.mean()
    return tab


display(preistabelle())


### Preisvergleich als Balkendiagramm

Das Balkendiagramm stellt dieselben Ergebnisse wie Tabelle 9 dar. Verändern Sie Wind- und Photovoltaikeinspeisung mit den Reglern. Vergleichen Sie sowohl die Höhe der Preissenkung als auch ihre Unterschiede zwischen den sechs Zeitschritten.


In [ ]:
def zeige_preise_im_tagesverlauf(wind=1200, pv=800, co2_preis=25):
    tab = preistabelle(wind, pv, co2_preis).drop(index="Mittel")
    x = np.arange(len(tab))

    spalte_ohne = "Preis ohne erneuerbare Einspeisung (€/MWh)"
    spalte_mit = "Preis mit Wind und PV (€/MWh)"
    spalte_diff = "Differenz (€/MWh)"

    fig, ax = plt.subplots(figsize=(9.5, 5.0))
    ax.bar(x - 0.2, tab[spalte_ohne], width=0.38, color="#9e9e9e",
           edgecolor="white", label="ohne erneuerbare Einspeisung", zorder=3)
    ax.bar(x + 0.2, tab[spalte_mit], width=0.38, color="#08519c",
           edgecolor="white", label="mit Wind und PV", zorder=3)

    for xi, (p0, p1, dp) in enumerate(zip(tab[spalte_ohne], tab[spalte_mit], tab[spalte_diff])):
        ax.annotate(f"Δp = {de(dp, 1)}", xy=(xi, max(p0, p1) + 1.5), ha="center",
                    va="bottom", fontsize=9, color="#e6550d")

    ax.set_xticks(x)
    ax.set_xticklabels(tab.index)
    ax.set_xlabel("Zeitschritt", fontsize=10)
    ax.set_ylabel("Marktpreis (€/MWh)", fontsize=10)
    ax.set_ylim(0, max(tab[[spalte_ohne, spalte_mit]].max()) * 1.24)
    ax.set_title(f"Marktpreise bei {de(wind, 0)} MW Wind und {de(pv, 0)} MW PV "
                 f"({de(co2_preis, 0)} €/tCO₂)", fontsize=11)
    ax.grid(False)
    ax.set_axisbelow(True)
    ax.set_facecolor("white")

    legend = ax.legend(loc="upper right", bbox_to_anchor=(0.985, 0.975),
                       frameon=True, fontsize=9, borderaxespad=0.0, fancybox=True,
                       markerfirst=False, handletextpad=0.8)
    frame = legend.get_frame()
    frame.set_facecolor("white"); frame.set_edgecolor("#bdbdbd")
    frame.set_linewidth(0.7); frame.set_alpha(0.96)
    set_boxstyle = getattr(frame, "set_boxstyle", None)
    if callable(set_boxstyle):
        set_boxstyle("round,pad=0.35,rounding_size=0.18")
    frame.set_path_effects([pe.SimplePatchShadow(offset=(1.5, -1.5), alpha=0.18,
                                                 shadow_rgbFace="#999999"), pe.Normal()])
    for txt in legend.get_texts():
        txt.set_horizontalalignment("right")
    if hasattr(legend, "set_alignment"):
        legend.set_alignment("right")

    fig.tight_layout()
    plt.show()

    print(f"Durchschnittlicher Preis ohne erneuerbare Einspeisung: {de(tab[spalte_ohne].mean())} €/MWh")
    print(f"Durchschnittlicher Preis mit erneuerbarer Einspeisung: {de(tab[spalte_mit].mean())} €/MWh")


interaktiv(zeige_preise_im_tagesverlauf,
           festes_beispiel=dict(wind=1200, pv=800, co2_preis=25),
           wind=(0, 4000, 50, 1200, "Wind (MW)"),
           pv=(0, 4000, 50, 800, "PV (MW)"),
           co2_preis=(0, 150, 5, 25, "CO₂-Preis (€/t)"))


## 9  CO₂-Preis und Brennstoffwechsel (Abbildung 13)

Ein CO₂-Preis erhöht die Grenzkosten einer Anlage proportional zu ihrem Emissionsfaktor. Emissionsintensive Kraftwerke werden dadurch stärker belastet und können innerhalb der Merit Order nach rechts rücken.

Zur Bestimmung des Wechselpreises zwischen Braunkohle und dem Gas-und-Dampf-Kraftwerk werden ihre Grenzkosten gleichgesetzt:

$$
12{,}25 + 1{,}05\,p^{\mathrm{CO_2}}
= 34{,}11 + 0{,}35\,p^{\mathrm{CO_2}}
$$

Daraus folgt:

$$
p_{\mathrm{CO_2}}^{\mathrm{Wechsel}}
=\frac{34{,}11-12{,}25}{1{,}05-0{,}35}
=31{,}23\;\text{€/tCO}_2
$$

Unterhalb dieses Preises weist die Braunkohle niedrigere Grenzkosten auf. Oberhalb des Wechselpreises rückt das Gas-und-Dampf-Kraftwerk vor die Braunkohle. Dieser Brennstoffwechsel verändert kurzfristig den Einsatz des bestehenden Kraftwerksparks, ohne dass zunächst neue Erzeugungskapazitäten errichtet werden müssen.


In [ ]:
# Wechselpreis unmittelbar aus den Eingabedaten beider Anlagen
a, b = TECH.loc["braunkohle"], TECH.loc["gud"]
p_wechsel = (b["c_var0"] - a["c_var0"]) / (a["ef"] - b["ef"])
print(f"Wechselpreis Braunkohle und Gas GuD: {de(p_wechsel)} €/tCO₂\n")

for co2 in [0, 25, p_wechsel, 60, 80]:
    mc = grenzkosten(co2)
    diff = mc["gud"] - mc["braunkohle"]
    if abs(diff) < 1e-6:
        befund = "gleiche Grenzkosten – Wechselpunkt"
    else:
        befund = "günstiger: " + ("Gas GuD" if diff < 0 else "Braunkohle")
    print(f"CO₂-Preis {de(co2):>6s} €/tCO₂  →  Braunkohle {de(mc['braunkohle']):>6s} €/MWh | "
          f"Gas GuD {de(mc['gud']):>6s} €/MWh  →  {befund}")

print("\nZum Vergleich: Der durchschnittliche Auktionspreis im EU-Emissionshandel lag 2024 "
      "bei 64,74 €/tCO₂. Auch die für das erste Halbjahr 2025 berichteten Preise lagen "
      "oberhalb des berechneten Wechselpreises (European Commission, 2025).")


In [ ]:
# Abbildung 13: Merit Order bei CO₂-Preisen von 25 und 80 €/tCO₂
fig, axes = plt.subplots(2, 1, figsize=(9.5, 9.4))

abbildung_11(nachfrage=4000, co2_preis=25, ax=axes[0], ymax=105)
axes[0].set_title("CO₂-Preis: 25 €/tCO₂", fontsize=11)

abbildung_11(nachfrage=4000, co2_preis=80, ax=axes[1], ymax=155)
axes[1].set_title("CO₂-Preis: 80 €/tCO₂", fontsize=11)

# Die Preisbeschriftung wird in beiden Feldern oberhalb der Preislinie platziert.
for ax in axes:
    preislinien = [ln for ln in ax.lines
                   if ln.get_color() == "#e6550d" and ln.get_linestyle() == ":"]
    if not preislinien:
        continue
    y_preis = preislinien[0].get_ydata()[0]
    for txt in ax.texts:
        if "Marktpreis p =" in txt.get_text():
            x_alt, _ = txt.get_position()
            txt.set_position((x_alt, y_preis + 2.5))
            txt.set_va("bottom")
            break

plt.tight_layout()
plt.show()


## 10  Marktwert und Selbstkannibalisierung (Abbildung 14)

Der Marktwert $MW_i$ ist der erzeugungsgewichtete Durchschnittspreis einer Technologie $i$. Dabei bezeichnet $p_t$ den Marktpreis, $g_{i,t}$ die Erzeugungsleistung und $\Delta t$ die Dauer des Zeitschritts:

$$
MW_i=\frac{\sum_t p_t\cdot g_{i,t}\cdot\Delta t}
            {\sum_t g_{i,t}\cdot\Delta t}
$$

Der zeitgewichtete durchschnittliche Marktpreis berücksichtigt dagegen alle Zeitschritte:

$$
\overline{p}=\frac{\sum_t p_t\cdot\Delta t}{\sum_t\Delta t}
$$

Der Wertigkeitsfaktor setzt den Marktwert ins Verhältnis zum durchschnittlichen Marktpreis:

$$
WF_i=\frac{MW_i}{\overline{p}}
$$

Ein Wertigkeitsfaktor über eins bedeutet, dass die Technologie überwiegend in vergleichsweise teuren Zeitschritten erzeugt. Ein Wert unter eins zeigt an, dass ihr Marktwert unter dem Marktdurchschnitt liegt.

Im Zahlenbeispiel erzeugt die Photovoltaik in den Zeitschritten t3, t4 und t5. Ihr Erzeugungsprofil und der konventionelle Kraftwerkspark bleiben unverändert. Variiert wird ausschließlich die installierte PV-Leistung.


In [ ]:
PV_ZEITSCHRITTE = ["t3", "t4", "t5"]

def wertigkeitsfaktor(pv_leistung, co2_preis=CO2_BASE):
    """Berechnet Marktwert und Wertigkeitsfaktor der Photovoltaik."""
    preise, erzeugung = {}, {}
    for t, d in DEMAND.items():
        pv_jetzt = pv_leistung if t in PV_ZEITSCHRITTE else 0.0
        preise[t], _ = marktpreis(d, co2_preis, wind=0.0, pv=pv_jetzt)
        erzeugung[t] = pv_jetzt
    preise = pd.Series(preise, dtype=float)
    erzeugung = pd.Series(erzeugung, dtype=float)

    p_mittel = (preise * DT).sum() / (len(preise) * DT)
    energie = float((erzeugung * DT).sum())
    if energie <= 0:
        return preise, p_mittel, float("nan"), float("nan")
    marktwert = (preise * erzeugung * DT).sum() / energie
    return preise, p_mittel, marktwert, marktwert / p_mittel


for leistung in [200, 1500]:
    preise, p_mittel, marktwert, wf = wertigkeitsfaktor(leistung)
    print(f"Installierte PV-Leistung: {leistung:5.0f} MW")
    print(f"  Marktpreise je Zeitschritt : {[de(p) for p in preise]}")
    print(f"  Durchschnittlicher Preis   : {de(p_mittel)} €/MWh")
    print(f"  Marktwert der PV            : {de(marktwert)} €/MWh")
    print(f"  Wertigkeitsfaktor           : {de(wf)}\n")


Der Wertigkeitsfaktor sinkt von **1,03 auf 0,87**, obwohl sich weder die Photovoltaiktechnologie noch ihr Erzeugungsprofil oder der konventionelle Kraftwerkspark verändern. Bei 1.500 MW verringert die zusätzliche Photovoltaikeinspeisung die Marktpreise gerade in den Zeitschritten, in denen die Photovoltaik erzeugt.

Dieser Rückgang des eigenen Erlöspreises wird als **Selbstkannibalisierung** bezeichnet. Er entspricht dem Merit-Order-Effekt aus der Perspektive der erneuerbaren Erzeuger. Die Investitionserlöse hängen deshalb vom technologiespezifischen Marktwert und nicht allein vom durchschnittlichen Marktpreis ab.

Die folgende Kurve reproduziert Abbildung 14 des Lehrtexts.


In [ ]:
# Abbildung 14: Wertigkeitsfaktor bei steigender installierter PV-Leistung
leistungen = np.arange(50, 2550, 50)
wfs = [wertigkeitsfaktor(leistung)[3] for leistung in leistungen]

fig, ax = plt.subplots(figsize=(9.5, 5.0))
ax.plot(leistungen, wfs, color="#08519c", linewidth=2.4, zorder=3)
ax.axhline(1.0, color="#666666", linestyle="--", linewidth=1.2, zorder=2)
ax.text(2500, 1.005, "Wertigkeitsfaktor = 1", color="#666666",
        fontsize=9, ha="right", va="bottom")

for leistung in [200, 1500]:
    wf = wertigkeitsfaktor(leistung)[3]
    ax.plot([leistung], [wf], marker="o", color="#e6550d", markersize=8, zorder=5)
    ax.annotate(f"{de(leistung, 0)} MW\nWF = {de(wf)}", xy=(leistung, wf),
                xytext=(leistung, wf - 0.01), color="#e6550d",
                fontsize=9.5, ha="center", va="top")

ax.set_xlabel("Installierte PV-Leistung (MW)", fontsize=10)
ax.set_ylabel("Wertigkeitsfaktor der PV (–)", fontsize=10)
ax.set_title("Wertigkeitsfaktor der Photovoltaik bei steigender installierter Leistung",
             fontsize=11)
ax.grid(False)
ax.set_axisbelow(True)
ax.set_facecolor("white")

plt.tight_layout()
plt.show()


### Marktwert und Durchschnittspreis vergleichen

Die folgende Abbildung zeigt die Marktpreise der sechs Zeitschritte. Gelbe Balken kennzeichnen die drei Zeitschritte mit Photovoltaikeinspeisung. In den grau dargestellten Zeitschritten erzeugt die Photovoltaik nicht. Die gestrichelte Linie zeigt den durchschnittlichen Marktpreis und die gepunktete Linie den Marktwert der Photovoltaik.

Erhöhen Sie die PV-Leistung schrittweise. Bei geringer Leistung liegt der Marktwert zunächst über dem durchschnittlichen Preis, da die Photovoltaik in Zeitschritten mit vergleichsweise hoher Nachfrage erzeugt. Mit zunehmender Leistung sinken die Preise in genau diesen Zeitschritten. Der Marktwert fällt dadurch schließlich unter den durchschnittlichen Marktpreis.

Zum Vergleich: In Deutschland betrug der allgemeine Jahresmarktwert 2025 89,32 €/MWh. Der Jahresmarktwert der Solarenergie lag bei 45,08 €/MWh. Daraus ergibt sich ein Wertigkeitsfaktor von rund 0,50 (Netztransparenz, 2026).


In [ ]:
def zeige_erloespreis(pv_leistung=1500):
    preise, p_mittel, marktwert, wf = wertigkeitsfaktor(pv_leistung)

    fig, ax = plt.subplots(figsize=(9.5, 5.0))
    farben = ["#fdd835" if t in PV_ZEITSCHRITTE else "#c8c8c8" for t in preise.index]
    ax.bar(preise.index, np.asarray(preise.to_numpy(), dtype=float), color=farben,
           edgecolor="white", linewidth=0.8, zorder=3)
    for t, p in preise.items():
        x = preise.index.tolist().index(t)
        ax.annotate(de(p), xy=(float(x), float(p)), xytext=(0, -4),
                    textcoords="offset points", ha="center", va="top",
                    fontsize=8.5, color="#555555")

    ax.axhline(float(p_mittel), color="#666666", linestyle="--", linewidth=1.6, zorder=4)
    ax.axhline(marktwert, color="#e6550d", linestyle=":", linewidth=2.2, zorder=4)
    ax.annotate(f"Durchschnittspreis  {de(float(p_mittel))}",
                xy=(5.45, float(p_mittel)), color="#666666", fontsize=9,
                va="bottom", ha="right")
    ax.annotate(f"Marktwert der PV  {de(marktwert)}", xy=(5.4, marktwert - 0.5),
                color="#e6550d", fontsize=9, va="top", ha="right")

    if np.isclose(wf, 1.0):
        vergleich = "entspricht dem"
    elif wf > 1:
        vergleich = "liegt über dem"
    else:
        vergleich = "liegt unter dem"
    ax.set_title(f"{de(pv_leistung, 0)} MW PV: WF = {de(wf)}; der Marktwert {vergleich} Durchschnittspreis",
                 fontsize=11)
    ax.set_ylabel("Marktpreis (€/MWh)", fontsize=10)
    ax.set_xlabel("Zeitschritt (gelb: mit PV-Einspeisung)", fontsize=10)
    ax.set_ylim(0, 90)
    ax.grid(axis="y", color="#ececec", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor("white")
    plt.tight_layout()
    plt.show()

interaktiv(zeige_erloespreis,
           festes_beispiel=dict(pv_leistung=1500),
           pv_leistung=(50, 2500, 50, 1500, "PV-Leistung (MW)"))


## 11  Zusammenfassung und Modellgrenzen

Mit diesem Notebook lassen sich die zentralen Mechanismen des Lehrtexts nachvollziehen:

1. Der Marktpreis entspricht den Grenzkosten der günstigsten Anlage, die noch über freie Kapazität verfügt. Bei Einheitspreisbildung erzielen Anlagen mit niedrigeren Grenzkosten inframarginale Renten (Abschnitte 4 bis 6).
2. Erneuerbare Einspeisung mit Grenzkosten nahe null verschiebt den konventionellen Teil der Angebotskurve nach rechts. Die resultierende Preissenkung hängt davon ab, welche Stufen der Merit Order überschritten werden (Abschnitte 7 und 8).
3. Der CO₂-Preis verändert die Grenzkosten unterschiedlich stark und kann dadurch die Einsatzreihenfolge der Kraftwerke verändern (Abschnitt 9).
4. Derselbe Merit-Order-Effekt senkt bei zunehmender zeitgleicher Einspeisung den Marktwert der Photovoltaik und führt zur Selbstkannibalisierung (Abschnitt 10).

**Modellgrenzen:** Alle Gebote im Notebook sind nichtnegativ und sämtliche Anlagen können ihre Erzeugung ohne zusätzliche Kosten verringern. Der Marktpreis kann deshalb bis auf null sinken, aber nicht negativ werden. Negative Preise würden unter anderem negative Gebote, Mindestlasten, Anfahrkosten oder erzeugungsabhängige Förderzahlungen erfordern. Speicher sind ebenfalls nicht enthalten. Die in Abschnitt 2.3.5 des Lehrtexts erläuterte Preisbildung über intertemporale Opportunitätskosten lässt sich daher mit diesem Notebook nicht abbilden.

**Eigene Erkundungen**

- Berechnen Sie den CO₂-Preis, bei dem Gas 1 geringere Grenzkosten als Steinkohle 2 aufweist. Prüfen Sie den Wechsel anschließend mit `grenzkosten()`.
- Ermitteln Sie, wie viel Windenergie im Zeitschritt t3 eingespeist werden muss, damit die Kernkraft preissetzend wird. Verwenden Sie dazu die Regler aus Abschnitt 7.
- Ergänzen Sie `PV_ZEITSCHRITTE` testweise um t2 und führen Sie Abschnitt 10 erneut aus. Wie verändert sich der Wertigkeitsfaktor?
- Vergleichen Sie eine Nachfrage von genau 4.320 MW mit einer Nachfrage oberhalb von 4.320 MW. Erläutern Sie den Unterschied zwischen vollständiger Kapazitätsauslastung und einer nicht mehr deckbaren Nachfrage.
